In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pcaDat = pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/PCA/0925/plot/KNC_EIA_IBD.csv',sep=',',
                    names=['Name','PC1','PC2','PC3','PC4','Population'],dtype={'Name':str},
                     skiprows=1, skipinitialspace=True)
print(pcaDat)

In [ ]:
symbolVec = ['8','s','p','P','*','h','H','X','D','d',',','<','>','^','v','o','.','$a$','$b$','$c$','$d$','$e$',
'$f$','$g$','$h$','$i$','$j$','$k$','$l$','$m$','$n$','$o$','$p$','$q$','$r$','$s$','$t$','$u$','$v$','$w$','$x$','$y$','$z$','$A$','$B$','$C$','$D$','$E$',
'$F$','$G$','$H$','$I$','$J$','$K$','$L$','$M$','$N$','$O$','$P$','$Q$','$R$','$S$','$T$','$U$','$V$','$W$','$X$','$Y$','$Z$','+','x','1','2','3','4','|','_']
colorVec = [u'#1f77b4', u'#ff7f0e', u'#f72585', u'#d62728', u'#9467bd',u'#023e8a',
            u'#8c564b', u'#e377c2', u'#bcbd22', u'#17becf',u'#003049',u'#730220',u'#ffb703',u'#9e0059',u'#ffff3f',u'#2b9348',u'#219ebc',u'#ffd60a']

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.patches as mpatches



# Read TSV (iid, cts)
cts_df = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/PCA/0925/plot/ibd_cts_phase3_from2.tsv", sep="\t")
cts_df["iid"] = cts_df["iid"].astype(str)
cts_df["cts"] = cts_df["cts"].astype(int)

iid2cts = dict(zip(cts_df["iid"], cts_df["cts"]))

# PCA iids
pca_iids = set(pcaDat["Name"].astype(str))
tsv_iids = set(cts_df["iid"])

missing_in_pca = sorted(tsv_iids - pca_iids)

print("Individuals in TSV but NOT in PCA (cannot plot):")
for iid in missing_in_pca:
    print(iid)

from matplotlib import cm

cts_values = sorted(cts_df["cts"].unique())
cts2idx = {c: i for i, c in enumerate(cts_values)}
K = len(cts_values)

# discrete viridis
cmap = cm.get_cmap("viridis", K)

def cts_to_color(cts):
    return cmap(cts2idx[cts])

legend_handles = []
legend_labels = []

for cts in cts_values:
    patch = mpatches.Patch(
        facecolor=cts_to_color(cts),
        edgecolor="k",
        label=str(cts)
    )
    legend_handles.append(patch)
    legend_labels.append(str(cts))

fig = plt.figure(figsize=(12, 12), dpi=600, facecolor="white")
ax = fig.add_subplot(111, facecolor="white")

ax.axis("equal")
ax.set_aspect("equal", "box")
ax.set(xlim=(-0.0014, 0.0013), ylim=(-0.0019, 0.0003))
ax.tick_params(axis='both', which='both',
               bottom=False, top=False, left=False, right=False,
               labelbottom=False, labelleft=False)

# Background (optional)
ax.scatter(
    pcaDat["PC1"], -pcaDat["PC2"],
    c="#e0e0e0", s=8, alpha=0.2, linewidths=0
)

# Plot TSV individuals, lower cts first (background), higher cts last (foreground)
for cts in sorted(cts_values):
    color = cts_to_color(cts)

    # all iids with this cts
    iids_this_cts = cts_df.loc[cts_df["cts"] == cts, "iid"].astype(str)

    for iid in iids_this_cts:
        d = pcaDat[pcaDat["Name"].astype(str) == iid]
        if d.empty:
            continue

        ax.scatter(
            d["PC1"], -d["PC2"],
            c=[color],
            s=100,
            ec="k",
            alpha=0.9
        )


ax.legend(
    handles=legend_handles,
    title="Count",
    loc="upper right",      # or (1.05, 0) for outside
    fontsize=18,
    title_fontsize=20,
    frameon=False,
    ncol=2
)
plt.xlabel('PC1 (0.71%)', fontsize=20)
plt.ylabel('PC2 (0.40%)', fontsize=20)
plt.show()
